# Projeto 1 — Demand Drivers & Commercial Forecast

## Electronics e Personal Care

### Objetivo
Investigar se variáveis comerciais ajudam a explicar a demanda e construir um modelo de previsão condicionado a fatores como:

- preço;
- desconto;
- promoção;
- estoque;
- categoria;
- calendário.

Nesta fase, restringimos a análise a **Electronics** e **Personal Care**, categorias nas quais o forecast univariado apresentou uma dinâmica futura visualmente mais coerente com a oscilação histórica.

> **Importante:** associação não significa causalidade. O objetivo é avaliar capacidade explicativa e preditiva, não provar que uma variável comercial causou a variação de vendas.


## 1. Bibliotecas e leitura da base tratada


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ARQUIVO = "retail_sales_cleaned_project1.csv"

df = pd.read_csv(
    ARQUIVO,
    parse_dates=["date"]
)

categorias = ["Electronics", "Personal Care"]

base = (
    df[df["category"].isin(categorias)]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Linhas selecionadas: {len(base):,}")
print(f"Período: {base['date'].min().date()} a {base['date'].max().date()}")
display(base.head())


## 2. Diagnóstico da volatilidade semanal

Antes de relacionar picos de vendas com promoção, preço ou desconto, precisamos verificar se o total semanal pode estar sendo influenciado pela própria quantidade de observações existentes em cada semana.

Se uma semana possui mais linhas na base, naturalmente pode apresentar mais unidades vendidas quando usamos `sum(units_sold)`.

Por isso, calculamos:

- unidades totais;
- quantidade de registros;
- unidades médias por observação;
- preço médio;
- desconto médio;
- participação de promoções;
- estoque médio.


In [ ]:
weekly_diag = (
    base.set_index("date")
        .groupby("category")
        .resample("W-SUN")
        .agg(
            units_sold=("units_sold", "sum"),
            records=("units_sold", "size"),
            avg_units_per_record=("units_sold", "mean"),
            avg_price=("price", "mean"),
            avg_discount=("discount_percent", "mean"),
            avg_inventory=("inventory_level", "mean"),
            promotion_rate=(
                "promotion_active",
                lambda x: (x == "Yes").mean() * 100
            )
        )
        .reset_index()
)

display(weekly_diag.head())


### Total vendido × quantidade de registros

Este teste é importante para interpretar corretamente o gráfico semanal anterior.

Se a correlação entre `units_sold` e `records` for muito elevada, parte relevante dos picos e vales pode decorrer da quantidade de observações disponíveis em cada semana, e não necessariamente de uma mudança real de comportamento comercial.


In [ ]:
for categoria in categorias:
    temp = weekly_diag[weekly_diag["category"] == categoria]

    corr = temp[["units_sold", "records"]].corr().iloc[0, 1]

    print(
        f"{categoria}: correlação entre total semanal vendido "
        f"e quantidade de registros = {corr:.2f}"
    )

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["records"],
        temp["units_sold"],
        alpha=0.75
    )
    plt.title(f"Quantidade de registros x Total vendido — {categoria}")
    plt.xlabel("Registros na semana")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Decisão metodológica

Como o total semanal pode ser fortemente afetado pelo número de registros existentes na base, o modelo explicativo será construído no **nível de observação**.

Assim:

- variável-alvo: `units_sold`;
- cada linha continua representando uma observação comercial;
- os fatores comerciais são usados diretamente para prever a quantidade vendida;
- posteriormente, as previsões individuais são agregadas por semana para comparação com os valores observados.

Isso evita atribuir ao preço, desconto ou promoção uma volatilidade que pode ser causada apenas pela quantidade de linhas da amostra.


## 3. Investigação dos fatores comerciais


In [ ]:
base["promotion_flag"] = (
    base["promotion_active"]
    .eq("Yes")
    .astype(int)
)

drivers = [
    "units_sold",
    "price",
    "discount_percent",
    "inventory_level",
    "promotion_flag"
]

for categoria in categorias:
    print(f"\n### {categoria}")
    display(
        base.loc[base["category"] == categoria, drivers]
            .corr()
            .round(3)
    )


### Promoção × unidades vendidas

Comparamos a média de vendas entre observações com e sem promoção.

A diferença observada é uma associação descritiva e não uma estimativa causal do efeito da promoção.


In [ ]:
promotion_summary = (
    base.groupby(["category", "promotion_active"])
        .agg(
            observations=("units_sold", "size"),
            avg_units=("units_sold", "mean"),
            median_units=("units_sold", "median"),
            avg_discount=("discount_percent", "mean"),
            avg_price=("price", "mean")
        )
        .reset_index()
)

display(promotion_summary)


### Desconto × unidades vendidas


In [ ]:
discount_summary = (
    base.groupby(["category", "discount_percent"])
        .agg(
            observations=("units_sold", "size"),
            avg_units=("units_sold", "mean"),
            avg_price=("price", "mean")
        )
        .reset_index()
)

display(discount_summary)


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["discount_percent"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Desconto x Unidades vendidas — {categoria}")
    plt.xlabel("Desconto (%)")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Preço × unidades vendidas


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["price"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Preço x Unidades vendidas — {categoria}")
    plt.xlabel("Preço")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


### Estoque × unidades vendidas

O estoque disponível pode limitar a quantidade vendida. Ainda assim, uma correlação simples não é suficiente para concluir que estoque maior gera vendas maiores.


In [ ]:
for categoria in categorias:
    temp = base[base["category"] == categoria]

    plt.figure(figsize=(7, 5))
    plt.scatter(
        temp["inventory_level"],
        temp["units_sold"],
        alpha=0.55
    )
    plt.title(f"Estoque x Unidades vendidas — {categoria}")
    plt.xlabel("Nível de estoque")
    plt.ylabel("Unidades vendidas")
    plt.tight_layout()
    plt.show()


## 4. Picos e vales semanais

Agora observamos as semanas de maior e menor volume e os respectivos indicadores comerciais.

O objetivo é verificar se os extremos apresentam padrões semelhantes de promoção, desconto, preço ou estoque.


In [ ]:
for categoria in categorias:

    temp = (
        weekly_diag[weekly_diag["category"] == categoria]
        .sort_values("units_sold", ascending=False)
    )

    print(f"\n{categoria} — 5 maiores semanas")
    display(temp.head(5))

    print(f"{categoria} — 5 menores semanas")
    display(temp.tail(5))


## 5. Preparação das features para previsão

O modelo utilizará apenas informações que poderiam ser conhecidas ou planejadas comercialmente:

### Comerciais
- `price`
- `discount_percent`
- `inventory_level`
- `promotion_active`

### Contexto
- `category`
- `store_id`
- dia da semana
- mês
- fim de semana

`product_id` não será utilizado nesta fase porque existem muitos produtos com poucas observações, o que aumentaria muito a dimensionalidade e o risco de overfitting.


In [ ]:
model_data = base.copy()

model_data["month_num"] = model_data["date"].dt.month
model_data["day_name"] = model_data["date"].dt.day_name()
model_data["is_weekend_model"] = (
    model_data["date"].dt.dayofweek >= 5
).astype(int)

# Representação cíclica do mês
model_data["month_sin"] = np.sin(
    2 * np.pi * model_data["month_num"] / 12
)
model_data["month_cos"] = np.cos(
    2 * np.pi * model_data["month_num"] / 12
)

features = [
    "price",
    "discount_percent",
    "inventory_level",
    "promotion_active",
    "category",
    "store_id",
    "day_name",
    "is_weekend_model",
    "month_sin",
    "month_cos"
]

target = "units_sold"

X = model_data[features].copy()
y = model_data[target].copy()


## 6. Separação temporal treino/teste

Não utilizaremos divisão aleatória.

O conjunto de teste será formado pelas observações das últimas 8 semanas do histórico, preservando a ordem temporal e simulando uma previsão real de período futuro.


In [ ]:
cutoff_date = model_data["date"].max() - pd.Timedelta(weeks=8)

train_mask = model_data["date"] <= cutoff_date
test_mask = model_data["date"] > cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

meta_test = model_data.loc[
    test_mask,
    ["date", "category", "store_id"]
].copy()

print(f"Data de corte: {cutoff_date.date()}")
print(f"Treino: {len(X_train):,} observações")
print(f"Teste: {len(X_test):,} observações")


## 7. Modelos comerciais

Serão comparados dois modelos:

### Ridge Regression
Modelo linear regularizado, útil como referência interpretável.

### Random Forest
Modelo não linear capaz de capturar interações entre desconto, promoção, preço, estoque, loja e calendário.

O objetivo não é escolher o modelo mais complexo, mas verificar se os fatores comerciais oferecem capacidade preditiva relevante.


In [ ]:
numeric_features = [
    "price",
    "discount_percent",
    "inventory_level",
    "is_weekend_model",
    "month_sin",
    "month_cos"
]

categorical_features = [
    "promotion_active",
    "category",
    "store_id",
    "day_name"
]

preprocessor_linear = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

ridge_model = Pipeline([
    ("preprocessor", preprocessor_linear),
    ("model", Ridge(alpha=1.0))
])

rf_model = Pipeline([
    ("preprocessor", preprocessor_tree),
    (
        "model",
        RandomForestRegressor(
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        )
    )
])


In [ ]:
ridge_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

pred_ridge = ridge_model.predict(X_test)
pred_rf = rf_model.predict(X_test)


## 8. Métricas

Além do MAE, RMSE, MAPE e Bias, as métricas serão calculadas também por categoria.


In [ ]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            np.abs(
                (y_true[mask] - y_pred[mask])
                / y_true[mask]
            )
        )
        * 100
    )


def bias(y_true, y_pred):
    return np.mean(
        np.asarray(y_pred)
        - np.asarray(y_true)
    )


def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAPE": mape(y_true, y_pred),
        "Bias": bias(y_true, y_pred)
    }


In [ ]:
overall_results = pd.DataFrame([
    {
        "Model": "Ridge",
        **metrics(y_test, pred_ridge)
    },
    {
        "Model": "Random Forest",
        **metrics(y_test, pred_rf)
    }
])

display(overall_results)


In [ ]:
predictions = meta_test.copy()
predictions["actual"] = y_test.values
predictions["Ridge"] = pred_ridge
predictions["Random Forest"] = pred_rf

category_results = []

for categoria in categorias:

    temp = predictions[
        predictions["category"] == categoria
    ]

    for modelo in ["Ridge", "Random Forest"]:

        category_results.append({
            "Category": categoria,
            "Model": modelo,
            **metrics(
                temp["actual"],
                temp[modelo]
            )
        })

category_results = pd.DataFrame(category_results)

display(
    category_results
    .sort_values(["Category", "MAPE"])
)


## 9. Actual vs Predicted — agregado por semana

As previsões são feitas em nível de observação e depois agregadas semanalmente.

Esse gráfico permite comparar a forma da demanda observada com a demanda prevista sem utilizar diretamente o número de registros como variável explicativa.


In [ ]:
weekly_predictions = (
    predictions
    .groupby(["date", "category"], as_index=False)
    .agg(
        actual=("actual", "sum"),
        Ridge=("Ridge", "sum"),
        Random_Forest=("Random Forest", "sum")
    )
)

# A agregação acima ainda está por data.
# Reagregamos para semana.
weekly_predictions = (
    weekly_predictions
    .set_index("date")
    .groupby("category")
    .resample("W-SUN")
    .agg(
        actual=("actual", "sum"),
        Ridge=("Ridge", "sum"),
        Random_Forest=("Random_Forest", "sum")
    )
    .reset_index()
)

display(weekly_predictions.head())


In [ ]:
for categoria in categorias:

    temp = weekly_predictions[
        weekly_predictions["category"] == categoria
    ]

    plt.figure(figsize=(10, 4))

    plt.plot(
        temp["date"],
        temp["actual"],
        marker="o",
        label="Actual"
    )

    plt.plot(
        temp["date"],
        temp["Ridge"],
        marker="o",
        linestyle="--",
        label="Ridge"
    )

    plt.plot(
        temp["date"],
        temp["Random_Forest"],
        marker="o",
        linestyle="--",
        label="Random Forest"
    )

    plt.title(
        f"Actual vs Commercial Models — {categoria}"
    )
    plt.xlabel("Semana")
    plt.ylabel("Unidades vendidas")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Importância das variáveis — Random Forest

Usamos **Permutation Importance** diretamente sobre o conjunto de teste.

A medida responde:

> Quanto o desempenho do modelo piora quando embaralhamos uma determinada variável?

Isso ajuda a identificar quais informações tiveram maior utilidade preditiva no período de teste.

A importância continua sendo **preditiva**, não causal.


In [ ]:
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance": perm.importances_mean
    })
    .sort_values("importance", ascending=True)
)

display(
    importance.sort_values(
        "importance",
        ascending=False
    )
)


In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.barh(
    importance["feature"],
    importance["importance"]
)

plt.title(
    "Importância preditiva das variáveis — Random Forest"
)
plt.xlabel("Queda de desempenho após permutação")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 11. Investigação específica por categoria

Além da importância global, verificamos novamente os principais indicadores descritivos separadamente para Electronics e Personal Care.

Isso é importante porque a relação entre desconto e demanda pode ser diferente entre categorias.


In [ ]:
for categoria in categorias:

    temp = base[
        base["category"] == categoria
    ]

    print(f"\n{categoria}")

    resumo = pd.DataFrame({
        "Métrica": [
            "Unidades médias",
            "Preço médio",
            "Desconto médio",
            "% observações em promoção",
            "Estoque médio"
        ],
        "Valor": [
            temp["units_sold"].mean(),
            temp["price"].mean(),
            temp["discount_percent"].mean(),
            temp["promotion_flag"].mean() * 100,
            temp["inventory_level"].mean()
        ]
    })

    display(resumo)


# 12. Forecast condicionado a cenários comerciais

Há uma diferença importante entre um forecast univariado e este modelo:

### Holt-Winters
Pode projetar o futuro apenas a partir do comportamento passado.

### Modelo com fatores comerciais
Para prever o futuro, precisamos informar quais serão as condições comerciais futuras:

- preço;
- desconto;
- promoção;
- estoque;
- loja;
- calendário.

Por isso, esta etapa gera **previsões condicionais**, e não uma única previsão inevitável.

A seguir criamos uma estrutura de cenário que pode ser modificada manualmente.


In [ ]:
# Escolhemos o modelo comercial com menor MAPE geral
best_model_name = (
    overall_results
    .sort_values("MAPE")
    .iloc[0]["Model"]
)

best_model = (
    ridge_model
    if best_model_name == "Ridge"
    else rf_model
)

print("Modelo selecionado:", best_model_name)


## 13. Cenário-base para as próximas 8 semanas

Para cada categoria, usamos como referência:

- preço mediano recente;
- desconto mediano recente;
- estoque mediano recente;
- proporção recente de promoções.

Como o modelo trabalha em nível de observação e `store_id` faz parte das features, o cenário futuro é criado para todas as lojas presentes no histórico recente.

> Estes valores são **premissas de cenário**, não informações futuras conhecidas.


In [ ]:
FORECAST_WEEKS = 8
RECENT_WEEKS = 8

recent_cutoff = (
    base["date"].max()
    - pd.Timedelta(weeks=RECENT_WEEKS)
)

recent = base[
    base["date"] > recent_cutoff
].copy()

last_date = base["date"].max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=FORECAST_WEEKS * 7,
    freq="D"
)

scenario_rows = []

for categoria in categorias:

    cat_recent = recent[
        recent["category"] == categoria
    ]

    price_assumption = cat_recent["price"].median()
    discount_assumption = cat_recent[
        "discount_percent"
    ].median()
    inventory_assumption = cat_recent[
        "inventory_level"
    ].median()

    promo_share = (
        cat_recent["promotion_active"]
        .eq("Yes")
        .mean()
    )

    stores = sorted(
        cat_recent["store_id"]
        .unique()
    )

    for date in future_dates:
        for store in stores:

            # Cenário determinístico:
            # considera promoção quando a participação
            # histórica recente for >= 50%.
            promo = (
                "Yes"
                if promo_share >= 0.5
                else "No"
            )

            scenario_rows.append({
                "date": date,
                "category": categoria,
                "store_id": store,
                "price": price_assumption,
                "discount_percent": discount_assumption,
                "inventory_level": inventory_assumption,
                "promotion_active": promo
            })

future_scenario = pd.DataFrame(scenario_rows)

future_scenario["month_num"] = (
    future_scenario["date"].dt.month
)

future_scenario["day_name"] = (
    future_scenario["date"].dt.day_name()
)

future_scenario["is_weekend_model"] = (
    future_scenario["date"].dt.dayofweek >= 5
).astype(int)

future_scenario["month_sin"] = np.sin(
    2 * np.pi
    * future_scenario["month_num"]
    / 12
)

future_scenario["month_cos"] = np.cos(
    2 * np.pi
    * future_scenario["month_num"]
    / 12
)

display(future_scenario.head())


In [ ]:
X_future = future_scenario[features]

future_scenario["forecast_units"] = np.maximum(
    best_model.predict(X_future),
    0
)

future_weekly = (
    future_scenario
    .set_index("date")
    .groupby("category")
    .resample("W-SUN")
    .agg(
        forecast_units=("forecast_units", "sum"),
        assumed_price=("price", "mean"),
        assumed_discount=("discount_percent", "mean"),
        assumed_inventory=("inventory_level", "mean"),
        promo_rate=(
            "promotion_active",
            lambda x: (x == "Yes").mean() * 100
        )
    )
    .reset_index()
)

display(future_weekly)


## 14. Visualização do forecast condicionado


In [ ]:
for categoria in categorias:

    hist = weekly_diag[
        weekly_diag["category"] == categoria
    ].copy()

    future = future_weekly[
        future_weekly["category"] == categoria
    ].copy()

    plt.figure(figsize=(11, 4))

    plt.plot(
        hist["date"],
        hist["units_sold"],
        label="Histórico"
    )

    plt.plot(
        future["date"],
        future["forecast_units"],
        marker="o",
        linestyle="--",
        label=f"Forecast condicionado — {best_model_name}"
    )

    plt.axvline(
        hist["date"].max(),
        linestyle=":",
        label="Início do cenário"
    )

    plt.title(
        f"Forecast com fatores comerciais — {categoria}"
    )
    plt.xlabel("Semana")
    plt.ylabel("Unidades vendidas")
    plt.legend()
    plt.tight_layout()
    plt.show()


# 15. Como interpretar esta fase

Ao final deste notebook, temos três perguntas separadas:

### 1. O histórico semanal é realmente tão volátil?
A análise de `records` ajuda a verificar quanto da variação do total semanal decorre da quantidade de observações presentes na base.

### 2. Quais fatores comerciais possuem associação com as vendas?
A análise exploratória compara preço, desconto, promoção e estoque com `units_sold`.

### 3. Esses fatores ajudam a prever?
Ridge e Random Forest são avaliados em período futuro separado do treino.

Se o desempenho dos modelos comerciais superar ou complementar o Holt-Winters, teremos evidência de que fatores externos acrescentam informação preditiva.

Se não superarem, isso também é um resultado válido: significa que, nesta base, as variáveis disponíveis explicam apenas parte limitada da demanda.


## 16. Exportação

Os resultados serão salvos para as próximas fases:

- métricas gerais;
- métricas por categoria;
- previsões do período de teste;
- importância das variáveis;
- cenário futuro condicionado.

Esses arquivos podem alimentar o planejamento financeiro e os cenários do Projeto 1.


In [ ]:
overall_results.to_csv(
    "commercial_model_results_overall.csv",
    index=False,
    encoding="utf-8-sig"
)

category_results.to_csv(
    "commercial_model_results_by_category.csv",
    index=False,
    encoding="utf-8-sig"
)

predictions.to_csv(
    "commercial_model_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

importance.sort_values(
    "importance",
    ascending=False
).to_csv(
    "commercial_driver_importance.csv",
    index=False,
    encoding="utf-8-sig"
)

future_weekly.to_csv(
    "commercial_forecast_future_8_weeks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos gerados:")
print("- commercial_model_results_overall.csv")
print("- commercial_model_results_by_category.csv")
print("- commercial_model_test_predictions.csv")
print("- commercial_driver_importance.csv")
print("- commercial_forecast_future_8_weeks.csv")
